In [2]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------------
# Sample: the 21 complete SSA countries, minus the two stale-year outliers
# (Cabo Verde 2009, Sierra Leone 2014 — dropped per prior decision)
# ---------------------------------------------------------------
COMPLETE_19 = {
    "AGO": "Angola", "BWA": "Botswana", "BDI": "Burundi",
    "SWZ": "Eswatini", "ETH": "Ethiopia", "GMB": "Gambia, The", "GHA": "Ghana",
    "GNB": "Guinea-Bissau", "LSO": "Lesotho", "MWI": "Malawi", "MLI": "Mali",
    "NGA": "Nigeria", "RWA": "Rwanda", "SEN": "Senegal", "SYC": "Seychelles",
    "SDN": "Sudan", "TZA": "Tanzania", "UGA": "Uganda", "ZMB": "Zambia",
}

master = pd.DataFrame({"iso3": list(COMPLETE_19.keys()), "country": list(COMPLETE_19.values())})

# ---------------------------------------------------------------
# GAI exposure: total_exposure_share = (GRADIENT_TOTAL - GRADIENT_X) / GRADIENT_TOTAL
# computed separately for SEX_T, SEX_F, SEX_M, using each country's latest available year
# ---------------------------------------------------------------
gai = pd.read_csv("EMP_TEMP_SEX_GAI_NB_A-20260911T1441.csv.gz")
gai_19 = gai[gai["ref_area"].isin(COMPLETE_19.keys())].copy()

latest_year = (
    gai_19[(gai_19["classif1"] == "GAI_GRADIENT_TOTAL") & (gai_19["sex"] == "SEX_T")]
    .dropna(subset=["obs_value"])
    .groupby("ref_area")["time"].max()
)
gai_19 = gai_19[gai_19.apply(lambda r: r["time"] == latest_year.get(r["ref_area"], -1), axis=1)]

rows = []
for iso3, grp in gai_19.groupby("ref_area"):
    row = {"iso3": iso3, "exposure_year": latest_year[iso3]}
    for sex_code, label in [("SEX_T", "total"), ("SEX_F", "female"), ("SEX_M", "male")]:
        sub = grp[grp["sex"] == sex_code]
        total = sub.loc[sub["classif1"] == "GAI_GRADIENT_TOTAL", "obs_value"]
        x = sub.loc[sub["classif1"] == "GAI_GRADIENT_X", "obs_value"]
        row[f"{label}_exposure_share"] = (
            1 - (x.values[0] / total.values[0]) if len(total) and len(x) and total.values[0] > 0 else np.nan
        )
    rows.append(row)

exposure_df = pd.DataFrame(rows)
master = master.merge(exposure_df, on="iso3", how="left")
master["gender_gap"] = master["female_exposure_share"] - master["male_exposure_share"]

# ---------------------------------------------------------------
# AIPI overall score
# ---------------------------------------------------------------
aipi = pd.read_csv("aipi_overall_aipi.csv")[["iso3", "AI_PI"]]
master = master.merge(aipi[aipi["iso3"].isin(COMPLETE_19.keys())], on="iso3", how="left")

# ---------------------------------------------------------------
# Standardize within-sample, compute the Adjustment Gap
# ---------------------------------------------------------------
zscore = lambda s: (s - s.mean()) / s.std(ddof=0)
master["z_exposure"] = zscore(master["total_exposure_share"])
master["z_aipi"] = zscore(master["AI_PI"])
master["adjustment_gap"] = master["z_exposure"] - master["z_aipi"]

# ---------------------------------------------------------------
# Quadrant typology
# ---------------------------------------------------------------
def quadrant(row):
    high_exp, high_cap = row["z_exposure"] >= 0, row["z_aipi"] >= 0
    if high_exp and not high_cap:  return "High exposure / Low capacity (highest concern)"
    if high_exp and high_cap:      return "High exposure / High capacity (well-matched)"
    if not high_exp and high_cap:  return "Low exposure / High capacity (well-matched)"
    return "Low exposure / Low capacity"

master["quadrant"] = master.apply(quadrant, axis=1)
master = master.sort_values("adjustment_gap", ascending=False).reset_index(drop=True)
master.insert(0, "rank", range(1, len(master) + 1))

master.to_csv("ssa_adjustment_gap.csv", index=False)
print(master[["rank", "iso3", "country", "exposure_year", "total_exposure_share",
              "gender_gap", "AI_PI", "adjustment_gap", "quadrant"]].to_string(index=False))

FileNotFoundError: [Errno 2] No such file or directory: 'aipi_overall_aipi.csv'